# Autogen

Microsoft's agent framework
- Designed for multi-agent conversation
- Patterns of agent interaction
- Supports tool calling, code execution

Other agent frameworks: CrewAI, Swarm, LangGraph

**Software versions:** Python 3.10+, Autogen 0.4.0+

## Setup

Autogen install

In [1]:
#!pip install autogen

Installing LLM clients, you can use many LLM providers:
- OpenAI
- Azure
- Claude
- Gemini
- Ollama

We will be using Ollama to run open-source models locally.

In [2]:
#!pip install "autogen-ext[ollama]"

In [3]:
#!pip install "autogen-ext[openai]"

In [4]:
# chat client - openai
from autogen_ext.models.openai import OpenAIChatCompletionClient

openai_client = OpenAIChatCompletionClient(
    model="gpt-4o",
    api_key="OPENAI_API_KEY"
)

If you use an API key, make sure its secure! Don't put it in code and accidentally leak it, use environment variables, for example:

In [5]:
# export OPENAI_API_KEY=MY_API_KEY

import os
api_key = os.getenv("OPENAI_API_KEY")

In [6]:
# ollama client
from autogen_ext.models.ollama import OllamaChatCompletionClient

ollama_client = OllamaChatCompletionClient(
    model="mistral:latest",
    seed=42
)

## Autogen example
Simple Coding example with a coder and reviewer

In [7]:
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent

In [8]:
#create some agents
coder = AssistantAgent(
    name="Coder",
    model_client=ollama_client,
    system_message=("You are a helpful Python programmer. Write short Python functions to solve the given task.")     
)

reviewer = AssistantAgent(
    name="Reviewer",
    model_client=ollama_client,
    system_message=(
        "You are a strict code reviewer. Inspect the Coder's output, identify bugs or styling issues."
        "Output your comments and let the Coder make further changes to fix the issues"
        "Only output 'TERMINATE' when no more improvements are needed from the Coder and the conversation can end."
    )
)

In [9]:
# set a termination condition
from autogen_agentchat.conditions import TextMentionTermination

termination = TextMentionTermination("TERMINATE")

There are many kinds of termination messages (https://microsoft.github.io/autogen/stable//user-guide/agentchat-user-guide/tutorial/termination.html)

- Max messages
- Text mention
- Token usage
- Timeout (duration in seconds)
- Source match - after a particular agent responds
- Function call - when a tool call is executed with the matching name

Text mention termination can sometimes be **unreliable** (especially with smaller models, we might see an example of it now), but provides models with a means to end the chat when the task is complete.

In [10]:
# setup the group chat
from autogen_agentchat.teams import RoundRobinGroupChat

groupchat = RoundRobinGroupChat(
    [coder, reviewer], # list of agents
    termination_condition=termination,
    max_turns=10
)

There are a few different group chat "patterns":
- **Round Robin**: participants take turns one by one
- **Selector**: one agent acts as a selector to choose the next speaker based on the context and conversation so far
- **Swarm**: selects the next speaker based on handoff messages from one agent to another, effectively allowing the current acting agent to decide who should speak next.

### Run the task

In [11]:
from autogen_agentchat.ui import Console

recursion_task = "Write a Python function that returns the factorial of a number using recursion."

await Console(groupchat.run_stream(task=recursion_task))

---------- TextMessage (user) ----------
Write a Python function that returns the factorial of a number using recursion.
---------- TextMessage (Coder) ----------
 Sure, here's a simple Python function that calculates the factorial of a number using recursion:

```python
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)
```

This function works by checking if `n` is equal to 0. If it is, it returns 1 (since the factorial of 0 is 1). If `n` is not 0, it calls itself with the argument `n - 1`, and multiplies the result by `n`. This continues until `n` reaches 0, at which point the function returns the final result (the factorial of `n`).
---------- TextMessage (Reviewer) ----------
 Your code is correct and works as expected. However, for better readability, it's a good practice to add a base case for when `n` is less than 0, as this will prevent unexpected errors or runtime issues. Here's an updated version of your function:

```python
def fac

TaskResult(messages=[TextMessage(id='3a7c23e3-2fb3-4a7f-a2b5-ef177a34f5e3', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 11, 3, 0, 43, 20, 551379, tzinfo=datetime.timezone.utc), content='Write a Python function that returns the factorial of a number using recursion.', type='TextMessage'), TextMessage(id='66c1aa42-e88b-4296-b690-8ed61a5e2b01', source='Coder', models_usage=RequestUsage(prompt_tokens=40, completion_tokens=162), metadata={}, created_at=datetime.datetime(2025, 11, 3, 0, 43, 24, 417539, tzinfo=datetime.timezone.utc), content=" Sure, here's a simple Python function that calculates the factorial of a number using recursion:\n\n```python\ndef factorial(n):\n    if n == 0:\n        return 1\n    else:\n        return n * factorial(n-1)\n```\n\nThis function works by checking if `n` is equal to 0. If it is, it returns 1 (since the factorial of 0 is 1). If `n` is not 0, it calls itself with the argument `n - 1`, and multiplies the result by `n`

### Multi-Agent Collaboration with Faulty Agents (Huang et al. ICML 2025)

Let's try something more complex with more agents

In [12]:
# Lead coder: writes clean code
lead_coder = AssistantAgent(
    name="LeadCoder",
    model_client=ollama_client,
    system_message=(
        "You are the lead coder. Write clean, correct Python code for the given task. "
    )
)

# Faulty coder, intentionally injects bugs
faulty_coder = AssistantAgent(
    name="FaultyCoder",
    model_client=ollama_client,
    system_message=(
        "You are a secondary coder who introduces faulty code. "
        "Sometimes you make small errors (e.g., wrong variable names, missing returns, poor styling choices). "
        "Do not mention or make comments on the faults you introduce. Just modify and send your version of the code."
    )
)

# Reviewer, finds and fixes issues
reviewer = AssistantAgent(
    name="Reviewer",
    model_client=ollama_client,
    system_message=(
        "You are a strict code reviewer. Inspect the Coder's output, identify bugs or styling issues."
        "Output your comments and let the Coder make further changes to fix the issues"
        "Only output 'TERMINATE' when no more improvements are needed from the Coder and the conversation can end."
    )
)

Note all the system messages. Each agent needs its own description on what its role is. These system messages can get much more detailed and complicated if needed, and is a easy first step to customise the agent system that you want.

In [13]:
termination = TextMentionTermination("TERMINATE")

faulty_chat = RoundRobinGroupChat(
    [lead_coder, faulty_coder, reviewer],
    termination_condition=termination,
    max_turns=20
)

In [14]:
await Console(faulty_chat.run_stream(task=recursion_task))

---------- TextMessage (user) ----------
Write a Python function that returns the factorial of a number using recursion.
---------- TextMessage (LeadCoder) ----------
 Here is a simple Python function that calculates the factorial of a number using recursion:

```python
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)
```

This function works by checking if the input number `n` is zero. If it is, it returns 1 (since the factorial of 0 is 1). Otherwise, it calls itself with the argument `n-1` and multiplies the result by `n`. This continues until it reaches the base case of `n=0`.

However, it's important to note that this function may lead to a StackOverflowError for large input values due to repeated function calls. In such cases, you can use an iterative approach or implement tail recursion optimization if your Python version supports it (Python 3.8 and above). Here is an example of the iterative approach:

```python
def factorial(n):
    

TaskResult(messages=[TextMessage(id='721dd1d7-100d-41e9-a91e-a89b74d7cb61', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 11, 3, 0, 43, 31, 797656, tzinfo=datetime.timezone.utc), content='Write a Python function that returns the factorial of a number using recursion.', type='TextMessage'), TextMessage(id='e9ca40ee-b257-4030-9f1c-746ab5b8110c', source='LeadCoder', models_usage=RequestUsage(prompt_tokens=41, completion_tokens=318), metadata={}, created_at=datetime.datetime(2025, 11, 3, 0, 43, 34, 472358, tzinfo=datetime.timezone.utc), content=" Here is a simple Python function that calculates the factorial of a number using recursion:\n\n```python\ndef factorial(n):\n    if n == 0:\n        return 1\n    else:\n        return n * factorial(n-1)\n```\n\nThis function works by checking if the input number `n` is zero. If it is, it returns 1 (since the factorial of 0 is 1). Otherwise, it calls itself with the argument `n-1` and multiplies the result by `n

# Customising Autogen - Intrinsic Memory Agents
What if we want to include some custom logic or memory to the agents?

The BaseChatAgent can be extended to a custom class, so we can add any additional features we want to use.

We need to fill in some abstract functions:
- on_messages(): what to do when getting incoming messages, returns a Response
- on_reset(): reset agent to initial state (clearing memory, cache, etc.)
- produced_message_types(): what kind of messages can be returned in a Response

In [15]:
from autogen_agentchat.agents import BaseChatAgent
from autogen_agentchat.messages import TextMessage, UserMessage
from autogen_core.model_context import UnboundedChatCompletionContext
from autogen_core.models import AssistantMessage
from autogen_agentchat.base import Response

class MemoryAgent(BaseChatAgent):

    def __init__(self, name, model_client, system_message, description):
        super().__init__(name=name, description=description)
        
        self._model_client = model_client
        self._conversation_history = UnboundedChatCompletionContext()
        self._memory_summary = ""
        self._initial_system_message = system_message
        self._current_system_message = system_message

    @property
    def produced_message_types(self):
        return (TextMessage,)

    async def on_reset(self, cancellation_token):
        self._conversation_history.clear()
        self._memory_summary = ""
        self._current_system_message = self._initial_system_message

    async def on_messages(self, messages, cancellation_token):
        # 1. Store messages in conversation history
        for msg in messages:
            await self._conversation_history.add_message(msg.to_model_message())

        # 2. Memory update
        await self._update_memory_summary()
        
        # 3. Build the context + memory
        model_messages = [
            msg.to_model_message() for msg in messages
        ]
        model_messages.append(
            UserMessage(content=self._current_system_message, source="system")
        )
        
        # 4. Call the model client for inference
        response = await self._model_client.create(model_messages)

        # get usage data
        # usage = RequestUsage(
        #     prompt_tokens=response.usage_metadata.prompt_token_count,
        #     completion_tokens=response.usage_metadata.candidates_token_count,
        # )

        await self._conversation_history.add_message(AssistantMessage(content=response.content, source=self.name))

        # 5. Yield the final response
        return Response(
            chat_message=TextMessage(content=response.content, source=self.name),# models_usage=usage),
            inner_messages=[],
        )


    async def _update_memory_summary(self):
        history_messages = await self._conversation_history.get_messages()

        if len(history_messages) == 0:
            return
        
        # get conversation history and put in one string of text
        history_text = "\n".join([
            f"[{msg.source}]: {msg.content if isinstance(msg.content, str) else str(msg.content)}"
            for msg in history_messages
            if hasattr(msg, 'content') and hasattr(msg, 'source')
        ])

        if not history_text:
            return

        # memory update prompt
        summarization_messages = [
            UserMessage(
                content="""Use the entire history of the conversation to 
                populate and update your current memory with factual information.
                Create a concise summary of the key points, decisions, and context from the conversation. 
                Focus on information that would be useful for future reference.""",
                source="system"
            ),
            UserMessage(
                content=f"Summarize the conversation history:\n\n{history_text}",
                source="user"
            )
        ]
        
        try:
            summary_result = await self._model_client.create(
                summarization_messages,
            )
            
            self._memory_summary = summary_result.content
            
            # update system message with memory
            self._update_system_message()
            
        except Exception as e:
            print(f"Failed to update memory summary: {e}")
    
    def _update_system_message(self):
        if self._memory_summary:
            self._current_system_message = (
                f"{self._initial_system_message}\n\n"
                f"MEMORY FROM PREVIOUS CONVERSATIONS:\n{self._memory_summary}"
            )

In [16]:
# final inspector using our custom MemoryAgent
memory_inspector = MemoryAgent(
    name="Inspector",
    model_client=ollama_client,
    system_message=(
        "You are the inspector. Given the final code and the original task, "
        "simulate running it mentally. If correct and readable, output 'TERMINATE'. "
        "Otherwise, request further fixes."
    ),
    description="Inspector with memory."
)

# change the reviewer to not output TERMINATE anymore
reviewer = AssistantAgent(
    name="Reviewer",
    model_client=ollama_client,
    system_message=(
        "You are a code reviewer. "
        "Review the last code, find logical or stylistic issues and identify them for the Coders to fix."
    )
)

In [17]:
memory_chat = RoundRobinGroupChat(
    [lead_coder, faulty_coder, reviewer, memory_inspector],
    termination_condition=termination,
    max_turns=20
)

In [18]:
await Console(memory_chat.run_stream(task=recursion_task))

---------- TextMessage (user) ----------
Write a Python function that returns the factorial of a number using recursion.
---------- TextMessage (LeadCoder) ----------
 Here's the clean and correct implementation of the recursive method for calculating factorials in Python:

```python
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)
```

This function takes an integer `n` as input and returns the factorial of that number. It uses recursion to calculate the factorial by checking if the input number `n` is equal to zero, in which case it returns 1 (since the factorial of 0 is 1). Otherwise, it calls itself with the argument `n-1` and multiplies the result by `n`, continuing until it reaches the base case of `n=0`.
---------- TextMessage (FaultyCoder) ----------
 ```python
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)
```
---------- TextMessage (Reviewer) ----------
 Your implementation of the fact

TaskResult(messages=[TextMessage(id='71d63eea-529a-4b47-aab9-6eaea702f2d3', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 11, 3, 0, 43, 49, 583953, tzinfo=datetime.timezone.utc), content='Write a Python function that returns the factorial of a number using recursion.', type='TextMessage'), TextMessage(id='8c6acd44-a2bb-4a81-ba2c-2b96822fb729', source='LeadCoder', models_usage=RequestUsage(prompt_tokens=1586, completion_tokens=164), metadata={}, created_at=datetime.datetime(2025, 11, 3, 0, 43, 51, 283415, tzinfo=datetime.timezone.utc), content=" Here's the clean and correct implementation of the recursive method for calculating factorials in Python:\n\n```python\ndef factorial(n):\n    if n == 0:\n        return 1\n    else:\n        return n * factorial(n-1)\n```\n\nThis function takes an integer `n` as input and returns the factorial of that number. It uses recursion to calculate the factorial by checking if the input number `n` is equal to zero, in

Hopefully we've seen some limitations of the termination message and round robin group chat:
- Might terminate early
- Might leak the termination message ("I will TERMINATE after the Coder fixes the errors")
- Round robin only allows one-way circular conversation

# Customising Autogen - Selector Group Chat

A selector group chat allows an LLM model to select which agent to go next, allowing the conversation to be more dynamic.

This comes with its own drawbacks, requiring a `selector_prompt` with instructions on how to select the next agent.

In [41]:
# Lead coder: writes clean code
lead_coder = AssistantAgent(
    name="LeadCoder",
    model_client=ollama_client,
    system_message=(
        "You are the lead coder. Write clean, correct Python code for the given task. "
    )
)

# Faulty coder, intentionally injects bugs
faulty_coder = AssistantAgent(
    name="FaultyCoder",
    model_client=ollama_client,
    system_message=(
        "You are a secondary coder who introduces faulty code. "
        "Sometimes you make small errors (e.g., wrong variable names, missing returns, poor styling choices). "
        "Do not mention or make comments on the faults you introduce. Just modify and send your version of the code."
    )
)

# Reviewer, finds and fixes issues
reviewer = AssistantAgent(
    name="Reviewer",
    model_client=ollama_client,
    system_message=(
        "You are a strict code reviewer. Inspect the Coder's output, identify bugs or styling issues."
        "Output your comments and let the Coder make further changes to fix the issues"
        "Only output 'TERMINATE' when no more improvements are needed from the Coder and the conversation can end."
    )
)

In [45]:
selector_prompt = """
Select an agent to go next.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to respond next.
The reviewer should generally go after the Coders to review their code.
Only select one agent.
"""

In [43]:
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.teams import SelectorGroupChat

text_termination = TextMentionTermination("TERMINATE")
max_messages_termination = MaxMessageTermination(max_messages=15)
termination = text_termination | max_messages_termination

selector_chat = SelectorGroupChat(
    [lead_coder, faulty_coder, reviewer],
    model_client=ollama_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=False,
)

In [44]:
await Console(selector_chat.run_stream(task=recursion_task))

---------- TextMessage (user) ----------
Write a Python function that returns the factorial of a number using recursion.
---------- TextMessage (LeadCoder) ----------
 Here is a simple Python function that calculates the factorial of a number using recursion:

```python
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)
```

This function works by checking if the input number `n` is zero. If it is, it returns 1 (since the factorial of 0 is 1). Otherwise, it calls itself with `n - 1`, multiplying the result by `n`. This continues until `n` reaches 0, at which point the recursion ends and the factorial is calculated.
---------- TextMessage (Reviewer) ----------
 While the provided function correctly calculates the factorial using recursion, there are a few stylistic improvements that could be made to enhance readability and adhere to some common Python coding conventions:

1. Indentation: In Python, the indentation level determines the nesting o

TaskResult(messages=[TextMessage(id='8d8d28af-bcbd-4f93-aa7e-4d7a96c10bea', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 11, 3, 0, 48, 43, 248547, tzinfo=datetime.timezone.utc), content='Write a Python function that returns the factorial of a number using recursion.', type='TextMessage'), TextMessage(id='5cbad4fa-d62e-4a6d-a0e1-f43fef1b4a23', source='LeadCoder', models_usage=RequestUsage(prompt_tokens=41, completion_tokens=145), metadata={}, created_at=datetime.datetime(2025, 11, 3, 0, 48, 44, 588407, tzinfo=datetime.timezone.utc), content=' Here is a simple Python function that calculates the factorial of a number using recursion:\n\n```python\ndef factorial(n):\n    if n == 0:\n        return 1\n    else:\n        return n * factorial(n-1)\n```\n\nThis function works by checking if the input number `n` is zero. If it is, it returns 1 (since the factorial of 0 is 1). Otherwise, it calls itself with `n - 1`, multiplying the result by `n`. This conti

Hopefully that was a helpful deepdive into using autogen to run a multi-agent system. There are still many pitfalls to be careful of, especially when using smaller language models:
- Sensitivity to prompts, requires specific and careful language, ideally use few-shot examples to guide the output
- More agents means harder to control output and intended termination/task completion
- Use custom agents and selectors to add more control

Find more details and further advanced tutorials in https://microsoft.github.io/autogen/stable//user-guide/agentchat-user-guide/tutorial/index.html